# Detectron2：云端同步与视频人物框验证

独立验证 Detectron2 的安装、GPU 算子和人物检测，不加载 SAM，不修改现有姿态提取流程。
可以随项目同步，也可以只上传本 notebook 到云端 GPU 平台。

1. 在云端选择已安装匹配版本 **PyTorch / torchvision** 的 Python kernel，推荐 Linux + NVIDIA GPU。
2. 修改配置，运行源码同步、依赖安装单元；已有环境可跳过安装。
3. 安装后**重启 kernel**，重新运行配置，再从环境检查单元继续。
4. 上传视频，修改 `VIDEO_PATH`，抽帧、加载检测器并运行检测与展示单元。

默认使用 COCO Faster R-CNN R50-FPN，均匀抽取 6 帧，仅展示 `person` 类及置信度。
**执行模型加载单元时才会下载官方权重到运行该 notebook 的机器（预期为云端）。**
本 notebook 交付前不在本地下载权重或执行真实检测。已有权重可通过 `WEIGHTS_PATH` 指定。


In [ ]:
from pathlib import Path
import os
import subprocess
import sys

WORK_DIR = Path.cwd().resolve()
RUNTIME_DIR = WORK_DIR / "runtime" / "detectron2_validation"
DETECTRON2_URL = "https://github.com/facebookresearch/detectron2.git"
DETECTRON2_REF = "main"  # 分支、tag 或 commit；首次验证后可填打印出的 commit 固定版本
DETECTRON2_DIR = RUNTIME_DIR / "detectron2"

DEVICE = "cuda"  # NVIDIA CUDA / AMD ROCm 均为 cuda；不会自动回退 CPU
MODEL_CONFIG = "COCO-Detection/faster_rcnn_R_50_FPN_3x.yaml"
WEIGHTS_PATH = None  # 已有云端权重时填 Path("/path/to/model_final_280758.pkl")

VIDEO_PATH = Path("input.mp4")  # 改为自己的视频；相对路径以 kernel 当前目录为准
NUM_FRAMES = 6
SCORE_THRESHOLD = 0.7
DISPLAY_COLS = 2

# Detectron2 的远程资源经 fvcore/iopath 缓存到本次运行目录。
os.environ.setdefault("FVCORE_CACHE", str(RUNTIME_DIR / "cache"))
os.environ.setdefault("TORCH_HOME", str(RUNTIME_DIR / "torch"))
print("Python:", sys.executable)
print("工作目录:", WORK_DIR)
print("Detectron2 源码:", DETECTRON2_DIR)


## 1. 同步 Detectron2 源码（首次准备时执行）

首次克隆官方仓库并 checkout 指定版本；已有目录只检查 origin、commit 和已跟踪文件的修改，
不自动 pull、reset 或覆盖。`main` 在首次克隆时解析，重复运行复用该次版本。
如需更新，选择新的 `DETECTRON2_DIR` 后重新准备；如需复现，将 `DETECTRON2_REF` 改为打印出的完整 commit。
仅上传 notebook 时也能执行，不需要先克隆 pose_pipeline。


In [ ]:
if not DETECTRON2_DIR.exists():
    DETECTRON2_DIR.parent.mkdir(parents=True, exist_ok=True)
    subprocess.run(
        ["git", "clone", "--no-checkout", DETECTRON2_URL, str(DETECTRON2_DIR)],
        check=True,
    )
    subprocess.run(
        ["git", "-C", str(DETECTRON2_DIR), "checkout", "--detach", DETECTRON2_REF],
        check=True,
    )
if not (DETECTRON2_DIR / ".git").exists():
    raise RuntimeError(f"目录不是 Git checkout，请更换 DETECTRON2_DIR: {DETECTRON2_DIR}")

git = ["git", "-C", str(DETECTRON2_DIR)]
origin = subprocess.check_output([*git, "remote", "get-url", "origin"], text=True).strip()
source_commit = subprocess.check_output([*git, "rev-parse", "HEAD"], text=True).strip()
expected_commit = subprocess.check_output(
    [*git, "rev-parse", f"{DETECTRON2_REF}^{{commit}}"], text=True,
).strip()
changes = subprocess.check_output(
    [*git, "status", "--porcelain", "--untracked-files=no"], text=True,
).strip()
if origin.rstrip("/") != DETECTRON2_URL or source_commit != expected_commit or changes:
    raise RuntimeError("源码来源、版本或修改状态不匹配；请检查配置或选择新的 DETECTRON2_DIR")
print("Detectron2 commit:", source_commit)


## 2. 使用当前 kernel 安装（已有环境可跳过）

本单元直接执行 pip；**不会安装或替换 PyTorch / torchvision**，需要事先准备匹配 GPU 的版本。
版本约束同时保护当前环境已安装的 torch/CUDA 相关包；其他依赖按 pip 正常解析，冲突时停止。
不读取 SAM 的 requirements.txt。

Detectron2 从源码编译，需要 Git、C/C++ 编译器，CUDA 构建还需要匹配的 CUDA toolkit / `nvcc`。
仅有 GPU 驱动或 PyTorch 自带 CUDA runtime 可能不足以编译。`--no-build-isolation` 让构建使用当前 torch。
`MAX_JOBS=2` 限制编译并发，可按云端内存调整。

**安装完成后重启 kernel，重新运行配置，然后跳过第 1、2 节，从第 3 节继续。**


In [ ]:
from importlib.metadata import distributions
import re

if not (DETECTRON2_DIR / "setup.py").is_file():
    raise FileNotFoundError("请先运行源码同步单元，或将 DETECTRON2_DIR 指向已有源码")
versions = {}
for dist in distributions():
    name = dist.metadata.get("Name")
    if name:
        versions[re.sub(r"[-_.]+", "-", name).lower()] = dist.version
if not all(name in versions for name in ("torch", "torchvision")):
    raise RuntimeError("请先为当前 GPU 安装匹配的 torch / torchvision，再运行此单元")
protected = {
    name: version for name, version in versions.items()
    if name in {"torch", "torchvision", "torchaudio", "torchtext", "torchdata", "triton"}
    or name.startswith(("nvidia-", "cuda-", "cudnn", "pytorch-triton", "torch-"))
}
RUNTIME_DIR.mkdir(parents=True, exist_ok=True)
constraints = RUNTIME_DIR / "torch-constraints.txt"
constraints.write_text(
    "".join(f"{name}=={version}\n" for name, version in sorted(protected.items())),
    encoding="utf-8",
)
pip = [sys.executable, "-m", "pip"]
dependencies = ["numpy>=1.26,<3", "matplotlib>=3.7", "ninja", "packaging", "setuptools<81", "wheel"]
if not any(name.startswith("opencv-") for name in versions):
    dependencies.append("opencv-python-headless>=4.8")
subprocess.run([*pip, "install", "-c", str(constraints), *dependencies], check=True)
build_env = dict(os.environ)
build_env.setdefault("MAX_JOBS", "2")
subprocess.run(
    [*pip, "install", "--no-build-isolation", "-c", str(constraints), "-e", str(DETECTRON2_DIR)],
    env=build_env, check=True,
)
subprocess.run([*pip, "check"], check=True)
print("安装完成。请重启 kernel，重跑配置，然后从第 3 节继续。")


## 3. 环境与 GPU 算子检查（不下载权重）

检查当前解释器实际导入的位置、版本、Detectron2 编译扩展与所选设备上的 NMS。
默认要求 GPU 可用；检查失败会停止，不以 CPU 结果代替 GPU 验证。
AMD ROCm 的 torch 设备名也为 `cuda`，但 Detectron2 扩展是否支持仍需在目标环境编译并验证。


In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision
from torchvision.ops import nms
import detectron2

try:
    from detectron2 import _C
except ImportError as exc:
    raise RuntimeError("Detectron2 编译扩展不可用；请在当前 torch/CUDA 环境重新构建并重启 kernel") from exc

device = torch.device(DEVICE)
if device.type not in {"cuda", "cpu"}:
    raise ValueError("本验证 notebook 的 DEVICE 仅支持 cuda 或 cpu")
if device.type == "cuda" and not torch.cuda.is_available():
    raise RuntimeError("GPU 不可用，请检查云端 GPU 分配、驱动及 torch 构建")
print("Python:", sys.version.split()[0], sys.executable)
print("torch:", torch.__version__, "torchvision:", torchvision.__version__)
print("CUDA:", torch.version.cuda, "ROCm:", torch.version.hip)
print("Detectron2:", detectron2.__version__, Path(detectron2.__file__).resolve())
print("编译扩展:", _C.__file__)
print("OpenCV:", cv2.__version__, "NumPy:", np.__version__)
if device.type == "cuda":
    print("GPU:", torch.cuda.get_device_name(device))
source_dir = Path(detectron2.__file__).resolve().parent.parent
if (source_dir / ".git").exists():
    print("Detectron2 commit:", subprocess.check_output(
        ["git", "-C", str(source_dir), "rev-parse", "HEAD"], text=True,
    ).strip())

probe_boxes = torch.tensor([[0, 0, 10, 10], [1, 1, 9, 9]], dtype=torch.float32, device=device)
probe_scores = torch.tensor([0.9, 0.8], dtype=torch.float32, device=device)
try:
    kept = nms(probe_boxes, probe_scores, 0.5).cpu().tolist()
except (RuntimeError, NotImplementedError) as exc:
    raise RuntimeError("所选设备的 torchvision NMS 不可用，请安装与 torch 匹配的 GPU torchvision") from exc
if kept != [0]:
    raise RuntimeError(f"NMS 检查结果异常: {kept}")
print(f"环境检查通过，NMS 实际运行设备: {device}")


## 4. 视频均匀抽帧

从视频第一帧到最后一帧均匀选取 `NUM_FRAMES` 帧；短视频不足时取全部可用帧。
OpenCV 按源帧号定位，仅读取抽样图像。这里的秒数是 `frame_index / fps` 的近似值，
适合检测功能检查，不作为变帧率视频的精确时间戳。
文件不存在、帧数/FPS 不可读或抽样解码失败时会明确报错。


In [ ]:
def sample_video_frames(video_path, num_frames):
    video_path = Path(video_path).expanduser().resolve()
    if not video_path.is_file():
        raise FileNotFoundError(f"请先上传视频并修改 VIDEO_PATH: {video_path}")
    if isinstance(num_frames, bool) or not isinstance(num_frames, int) or num_frames < 1:
        raise ValueError("NUM_FRAMES 必须是正整数")
    capture = cv2.VideoCapture(str(video_path))
    try:
        if not capture.isOpened():
            raise RuntimeError(f"OpenCV 无法打开视频: {video_path}")
        count_value = capture.get(cv2.CAP_PROP_FRAME_COUNT)
        fps = float(capture.get(cv2.CAP_PROP_FPS))
        if not np.isfinite(count_value) or count_value < 1 or not np.isfinite(fps) or fps <= 0:
            raise RuntimeError("无法读取有效帧数/FPS，请检查视频或转封装为普通 MP4 后重试")
        frame_count = int(count_value)
        indices = np.linspace(0, frame_count - 1, min(num_frames, frame_count), dtype=int)
        frames = []
        for frame_index in indices.tolist():
            if not capture.set(cv2.CAP_PROP_POS_FRAMES, frame_index):
                raise RuntimeError(f"无法定位到源帧 {frame_index}")
            ok, image_bgr = capture.read()
            if not ok or image_bgr is None:
                raise RuntimeError(f"源帧 {frame_index} 解码失败，请检查编码和视频完整性")
            frames.append({
                "frame_index": frame_index,
                "time_seconds_approx": frame_index / fps,
                "image_bgr": image_bgr,
            })
    finally:
        capture.release()
    return frames, {"path": str(video_path), "fps": fps, "frame_count": frame_count}


sampled_frames, video_info = sample_video_frames(VIDEO_PATH, NUM_FRAMES)
print("视频:", video_info)
print("抽样源帧号:", [frame["frame_index"] for frame in sampled_frames])
print("实际抽帧数:", len(sampled_frames))


## 5. 加载人物检测模型（在云端运行，会按需下载权重）

`WEIGHTS_PATH=None` 使用 Detectron2 model zoo 的官方 COCO 权重，并缓存在云端，重复使用缓存。
如果云端无法访问权重地址，先在云端准备对应配置的权重文件，再填写 `WEIGHTS_PATH`。
自定义权重需与 `MODEL_CONFIG` 的架构及 COCO 类别定义匹配。

COCO 在 Detectron2 中使用连续类别编号，`person=0`，与 torchvision 的 `person=1` 不同。
这里使用 Detectron2 独立预测器，不能将其权重路径直接传给现有 torchvision 检测器。


In [ ]:
from detectron2 import model_zoo
from detectron2.config import get_cfg
from detectron2.engine import DefaultPredictor

if not 0.0 <= SCORE_THRESHOLD <= 1.0:
    raise ValueError("SCORE_THRESHOLD 必须在 [0, 1] 内")
cfg = get_cfg()
cfg.merge_from_file(model_zoo.get_config_file(MODEL_CONFIG))
cfg.MODEL.DEVICE = str(device)
cfg.MODEL.ROI_HEADS.SCORE_THRESH_TEST = float(SCORE_THRESHOLD)
if WEIGHTS_PATH is None:
    cfg.MODEL.WEIGHTS = model_zoo.get_checkpoint_url(MODEL_CONFIG)
else:
    weights = Path(WEIGHTS_PATH).expanduser().resolve()
    if not weights.is_file():
        raise FileNotFoundError(weights)
    cfg.MODEL.WEIGHTS = str(weights)
print("配置:", MODEL_CONFIG)
print("权重:", cfg.MODEL.WEIGHTS)
print("设备:", cfg.MODEL.DEVICE)
predictor = DefaultPredictor(cfg)
print("模型加载完成")


## 6. 检测 person 并在 notebook 中展示

每帧只保留人物框；标签显示置信度，标题显示源帧号、近似时间和人数。
无人帧仍展示原图并标为 `people: 0`；模型异常会直接报错。
控制台保留 `detection_results` 的统计摘要，框坐标为原图像素 `xyxy`。
推理耗时含首帧预热开销，仅供排查运行状态。


In [ ]:
import math
import time
from matplotlib.patches import Rectangle
from IPython.display import display

if isinstance(DISPLAY_COLS, bool) or not isinstance(DISPLAY_COLS, int) or DISPLAY_COLS < 1:
    raise ValueError("DISPLAY_COLS 必须是正整数")
if not 0.0 <= SCORE_THRESHOLD <= 1.0:
    raise ValueError("SCORE_THRESHOLD 必须在 [0, 1] 内")
# 允许降低阈值后重跑本单元；同时更新模型内部阈值，避免只改变后处理阈值。
for box_predictor in predictor.model.roi_heads.box_predictor.modules():
    if hasattr(box_predictor, "test_score_thresh"):
        box_predictor.test_score_thresh = float(SCORE_THRESHOLD)

detection_results = []
for sample in sampled_frames:
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    started = time.perf_counter()
    outputs = predictor(sample["image_bgr"])
    instances = outputs["instances"]
    people = instances[
        (instances.pred_classes == 0) & (instances.scores >= SCORE_THRESHOLD)
    ].to("cpu")
    if device.type == "cuda":
        torch.cuda.synchronize(device)
    result = {
        "frame_index": sample["frame_index"],
        "time_seconds_approx": sample["time_seconds_approx"],
        "boxes_xyxy": people.pred_boxes.tensor.numpy().copy(),
        "scores": people.scores.numpy().copy(),
        "elapsed_seconds": time.perf_counter() - started,
    }
    detection_results.append(result)
    print(f"frame={result['frame_index']:>6}  time≈{result['time_seconds_approx']:.3f}s  "
          f"people={len(result['scores'])}  inference={result['elapsed_seconds']:.3f}s")

columns = min(DISPLAY_COLS, len(sampled_frames))
rows = math.ceil(len(sampled_frames) / columns)
figure, axes = plt.subplots(rows, columns, figsize=(8 * columns, 5 * rows), squeeze=False)
for axis in axes.flat:
    axis.axis("off")
for axis, sample, result in zip(axes.flat, sampled_frames, detection_results):
    axis.imshow(cv2.cvtColor(sample["image_bgr"], cv2.COLOR_BGR2RGB))
    for box, score in zip(result["boxes_xyxy"], result["scores"]):
        x1, y1, x2, y2 = box.tolist()
        axis.add_patch(Rectangle(
            (x1, y1), x2 - x1, y2 - y1, fill=False, edgecolor="lime", linewidth=2,
        ))
        axis.text(x1, max(0, y1 - 4), f"person {score:.2f}", color="black", fontsize=10,
                  bbox={"facecolor": "lime", "alpha": 0.85, "pad": 2, "edgecolor": "none"})
    axis.set_title(f"frame {result['frame_index']} | ~{result['time_seconds_approx']:.2f}s | "
                   f"people: {len(result['scores'])}")
figure.tight_layout()
display(figure)
plt.close(figure)


## 继续验证与常见问题

- **换视频或抽帧数**：在配置单元修改 `VIDEO_PATH` / `NUM_FRAMES`，重跑配置、抽帧和检测展示单元即可，模型继续复用。
- **调整阈值**：修改 `SCORE_THRESHOLD` 后重跑配置和检测展示单元即可；该单元同步更新 Faster R-CNN 的内部阈值。
- **更换设备、模型配置或权重**：重新运行环境检查与模型加载，再检测展示。
- **找不到 `_C` / undefined symbol**：核对实际导入路径、torch/CUDA 版本，在当前环境重新编译 Detectron2，并重启 kernel。
- **`torchvision::nms` 无 GPU 实现**：重新准备与 torch 匹配的 CUDA/ROCm torchvision；只改 `DEVICE` 无法修复 GPU 算子。
- **编译找不到 nvcc / CUDA_HOME 或内存不足**：换用含开发工具的云端镜像、配置 CUDA toolkit，或降低安装单元中的 `MAX_JOBS`。
- **所有抽样帧人数为 0**：先核对展示的帧是否有人，可降低阈值复测；无框本身不能证明模型故障或精度达标。

完整验证应看到：环境检查通过 → 实际抽帧号 → 模型成功加载 → 每帧统计 → 带人物框的图像。
这是独立的检测功能检查，不包含跟踪、姿态推理或 SAM 对接；真实 GPU 推理与检测效果由云端运行结果确认。
